In [27]:
import os
import random
paths = os.listdir("./training-tpdne")
paths = [os.path.join("./training-tpdne", p) for p in paths]

paths = sorted(paths)
random.seed(42)
random.shuffle(paths) 
paths = paths[:1000]

In [28]:
import numpy as np
import cv2
from insightface.app import FaceAnalysis

app = FaceAnalysis(name='antelopev2', root='./', providers=['CPUExecutionProvider'])


app.prepare(ctx_id=0, det_size=(160, 160))
def get_embedding(image_path):
    image = cv2.imread(image_path)
    if image is None:
        print("Failed to read image", image_path) 
        return None
    image = cv2.resize(image, (160,160))
    faces = app.get(image)
    if len(faces) == 0:
        print("No face found in", image_path)
        return None
    face = faces[0]
    embedding = face.embedding
    return embedding     

# use a pool of 64 cpus with tqdm
from multiprocessing import Pool
import tqdm

with Pool(130) as p:
    embeddings = list(tqdm.tqdm(p.imap(get_embedding, paths), total=len(paths)))



Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/arcface.onnx recognition ['None', 3, 112, 112] 127.5 127.5
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/scrfd_10g_bnkps.onnx detection [1, 3, '?', '?'] 127.5 128.0
set det-size: (160, 160)


100%|██████████| 1000/1000 [00:13<00:00, 72.29it/s]


In [29]:
import pickle
with open("tqdne_original_embeddings.pkl", "wb") as f:
    pickle.dump({
        "filenames": paths,
        "embeddings": embeddings,
        "images": [cv2.imread(p) for p in paths]
    }, f) 

In [30]:
paths = []
root = "/path/to/lfw"
for id in os.listdir(root):
    id_folder = os.path.join(root, id)
    if not os.path.isdir(id_folder):
        continue
    for file in os.listdir(id_folder):
        if file.endswith(".jpg"):
            paths.append(os.path.join(id_folder, file))
paths = sorted(paths)
random.seed(42)
random.shuffle(paths) 
paths = paths[:1000]

In [31]:
import numpy as np
import cv2
from insightface.app import FaceAnalysis

app = FaceAnalysis(name='antelopev2', root='./', providers=['CPUExecutionProvider'])


app.prepare(ctx_id=0, det_size=(160, 160))
def get_embedding(image_path):
    image = cv2.imread(image_path)
    if image is None:
        print("Failed to read image", image_path) 
        return None
    image = cv2.resize(image, (160,160))
    faces = app.get(image)
    if len(faces) == 0:
        print("No face found in", image_path)
        return None
    face = faces[0]
    embedding = face.embedding
    return embedding     

# use a pool of 64 cpus with tqdm
from multiprocessing import Pool
import tqdm

with Pool(130) as p:
    embeddings = list(tqdm.tqdm(p.imap(get_embedding, paths), total=len(paths)))

import pickle
with open("../../evaluation/results/holdout_original_embeddings.pkl", "wb") as f:
    pickle.dump({
        "filenames": paths,
        "embeddings": embeddings,
        "images": [cv2.imread(p) for p in paths]
    }, f) 


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/arcface.onnx recognition ['None', 3, 112, 112] 127.5 127.5
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: ./models/antelopev2/scrfd_10g_bnkps.onnx detection [1, 3, '?', '?'] 127.5 128.0
set det-size: (160, 160)


100%|██████████| 1000/1000 [00:18<00:00, 54.04it/s]


In [14]:
import pickle
with open("test.pkl", "rb") as f:
    data = pickle.load(f)

with open("../../attacks/minusface/test.pkl", "rb") as f:
    tqdne_data = pickle.load(f)

In [26]:
set([os.path.basename(i) for i in sum(data["filenames"], [])]) == set([os.path.basename(i) for i in sum(tqdne_data["filenames"], [])])

True